# Development of Arctic boundary current model

See `Development/TestingNavierStokes??.ipynb` series for development of NS solver in rectangular domain.

0. `ArcticBoundaryCurrentModel0.ipynb`: Adapt wind forced dishpan steady NS solver to Arctic boundary current geometry.
1. `ArcticBoundaryCurrentModel1.ipynb`: More work on gmsh domain. 
2. `ArcticBoundaryCurrentModel2.ipynb`: More work on gmsh domain. Add cooling and freshening patches to cylinder.
3. `ArcticBoundaryCurrentModel3.ipynb`: More work on gmsh domain. Add warming/salinification patch. But it still errors when defining the 3D domain (it only works on the surface of the gmsh domain).
4.  `ArcticBoundaryCurrentModel4.ipynb`: More work on gmsh domain. Try to get cylindrical disk to work. This succeeds.

twnh July '25

## Problem statement

The ultimate goal is to solve a nonlinear multi-field PDE. Consider here the lid-stress-driven flow for the incompressible rotating Navier-Stokes equations. Formally, the PDE we want to solve is: find the velocity vector $u$ and the pressure anomaly $p$ such that

$$
\left\lbrace
\begin{aligned}
-\nu  \nabla^2 u +  (u \nabla) u + (1/\rho_0) \nabla p + f  \hat{\mathbf k} \times u = 0 &\text{ in }\Omega,\\
\nabla\cdot u = 0 &\text{ in } \Omega,\\
\boldsymbol{t} \cdot \boldsymbol{\sigma} \cdot \mathbf{n} = \tau_{\text{imposed}} &\text{ on } \Gamma_s, \\
u = 0 &\text{ on } \Gamma_w,
\end{aligned}
\right.
$$

where the computational domain is the rectangle $\Omega \doteq (0,L_x) \times (-L_y/2,L_y/2) \times (-L_z,0)$, ${\mathbf n}$ is the unit outward normal, and $\boldsymbol{t}$ is the tangential direction at the surface, $\Gamma_s$. The driving force is the tangential stress $\tau_{\text{imposed}}$ (units of $\text{m}^{2} \text{s}^{-2}$ ). The mean value of the pressure anomaly is constrained to equal zero,

$$
\int_\Omega p \ {\rm d}\Omega = 0 .
$$


In [29]:
using Gridap
using GridapSolvers
using GridapSolvers.LinearSolvers, GridapSolvers.MultilevelTools, GridapSolvers.NonlinearSolvers
using GridapSolvers.BlockSolvers: LinearSystemBlock, NonlinearSystemBlock, BiformBlock, BlockTriangularSolver
using Gridap.MultiField
using GridapGmsh
using LinearAlgebra

#### Define physical parameters

In [30]:
# Physical properties
# ν = 1e-6              # (m^2/s) kinematic viscosity
ν = 1.0e-4              # (m^2/s) kinematic viscosity
ρₒ = 1000.0             # (kg/m^3) reference density
rotation_period = 6.0   # (s) rotation rate
f = 2*2*π/rotation_period
Ekman_layer_depth = sqrt(2*ν / f) # (m), Ekman layer depth
println("Ekman layer depth: ", Ekman_layer_depth, " m")

# Surface stress boundary condition
u₁₀(y) = 1.0     # m s⁻¹, average wind velocity 10 meters above the ocean
# u₁₀(y) =  1.0 .* cos(π.*y./Ly)     # m s⁻¹, average wind velocity 10 meters above the ocean
cᴰ = 2.5e-3 # dimensionless drag coefficient
ρₐ = 1.225  # kg m⁻³, average density of air at sea-level
Qᵘ(x) = VectorValue((ρₐ / ρₒ) * cᴰ * u₁₀(x[2]) * abs(u₁₀(x[2])),0,0) # m² s⁻²

Ekman layer depth: 0.009772050238058399 m


Qᵘ (generic function with 1 method)

Load model geometry.

In [31]:
model = GmshDiscreteModel("ArcticBasin4.msh")
# model = GmshDiscreteModel("testing/t16.msh")
labels = get_face_labeling(model)
label_names = model.face_labeling.tag_to_name

Info    : Reading 'ArcticBasin4.msh'...
Info    : 9 entities
Info    : 927 nodes
Info    : 4768 elements
Info    : Done reading 'ArcticBasin4.msh'


4-element Vector{String}:
 "Top"
 "Bottom"
 "Wall"
 "ArcticBasin"

## FE spaces

See: `Development/TestingNavierStokes??.ipynb` for more info on these spaces

In [32]:
order = 2
qdegree = 2*(order+1)
reffeᵤ = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
V = TestFESpace(model,reffeᵤ,conformity=:H1,labels=labels,dirichlet_tags=["Wall","Bottom"])
reffeₚ = ReferenceFE(lagrangian,Float64,order-1;space=:P)
Q = TestFESpace(model,reffeₚ,conformity=:L2,constraint=:zeromean)
uD0 = VectorValue(0,0,0)            # Velocity vanishes on the boundaries except the top
U = TrialFESpace(V,uD0)
P = TrialFESpace(Q)
mfs = Gridap.MultiField.BlockMultiFieldStyle()
Y = MultiFieldFESpace([V, Q];style=mfs)
X = MultiFieldFESpace([U, P];style=mfs)

MultiFieldFESpace()

## Triangulation and integration quadrature

From the discrete model we can define the triangulation and integration measure

In [33]:
degree = order
Ω = Triangulation(model)
dΩ = Measure(Ω,qdegree)
Γ = BoundaryTriangulation(model,tags="Top")
dΓ = Measure(Γ,degree)

GenericMeasure()

## Steady nonlinear Navier-Stokes solver with Coriolis force.

Use a preconditioned FGMRES solver. See: https://gridap.github.io/GridapSolvers.jl/stable/Examples/NavierStokes/

In [34]:
khat = VectorValue(0,0,1)
Coriolis(u,v,dΩ) = ∫(f * cross(khat, u) ⋅ v)dΩ # Coriolis term
b((v,q),dΓ) =  ∫(- v ⋅ Qᵘ)dΓ    # Boundary condition for the velocity

α = 1.e2    # Stabilization parameter
Π_Qh = LocalProjectionMap(divergence,Q,qdegree)
graddiv(u,v,dΩ) = ∫(α*(∇⋅v)⋅Π_Qh(u))dΩ

conv(u,∇u) = (∇u')⋅u
dconv(du,∇du,u,∇u) = conv(u,∇du)+conv(du,∇u)
c(u,v,dΩ) = ∫(v⊙(conv∘(u,∇(u))))dΩ
dc(u,du,dv,dΩ) = ∫(dv⊙(dconv∘(du,∇(du),u,∇(u))))dΩ

lap(u,v,dΩ) = ∫(ν*∇(v)⊙∇(u))dΩ

jac_u(u,du,dv,dΩ) = lap(du,dv,dΩ) + dc(u,du,dv,dΩ) + graddiv(du,dv,dΩ)
jac_u(u,du,dv,dΩ) = lap(du,dv,dΩ) + graddiv(du,dv,dΩ) + Coriolis(du,dv,dΩ)
jac((u,p),(du,dp),(dv,dq),dΩ) = jac_u(u,du,dv,dΩ) - ∫(divergence(dv)*dp)dΩ - ∫(divergence(du)*dq)dΩ

res_u(u,v,dΩ) = lap(u,v,dΩ) + c(u,v,dΩ) + graddiv(u,v,dΩ)
res_u(u,v,dΩ) = lap(u,v,dΩ) + graddiv(u,v,dΩ) + Coriolis(u,v,dΩ) 
res((u,p),(v,q),dΩ) = res_u(u,v,dΩ) - (1/ρₒ)*∫(divergence(v)*p)dΩ - (1/ρₒ)*∫(divergence(u)*q)dΩ + b((v,q),dΓ)

jac_h(x,dx,dy) = jac(x,dx,dy,dΩ)
res_h(x,dy) = res(x,dy,dΩ)
op = FEOperator(res_h,jac_h,X,Y)

solver_u = LUSolver()
solver_p = CGSolver(JacobiLinearSolver();maxiter=20,atol=1e-14,rtol=1.e-6,verbose=true)
solver_p.log.depth = 4

bblocks  = [NonlinearSystemBlock() LinearSystemBlock();
            LinearSystemBlock()    BiformBlock((p,q) -> ∫(-((ρₒ*ρₒ)/α)*p*q)dΩ,Q,Q)]
coeffs = [1.0 1.0;
          0.0 1.0]  
P = BlockTriangularSolver(bblocks,[solver_u,solver_p],coeffs,:upper)
solver = FGMRESSolver(20,P;atol=1e-11,rtol=1.e-8,verbose=true)
solver.log.depth = 2

nlsolver = NewtonSolver(solver;maxiter=20,atol=1e-10,rtol=1.e-12,verbose=true)
uh,ph = solve(nlsolver,op)

--------------- Starting Newton-Raphson solver --------
  > Iteration   0 - Residuals: 3.63e-07,   1.00e+00 
    --------------- Starting FGMRES solver ----------------
      > Iteration   0 - Residuals: 3.63e-07,   1.00e+00 
        --------------- Starting CG solver --------------------
          > Iteration   0 - Residuals: 0.00e+00,   1.00e+00 
        Solver CG finished with reason SOLVER_CONVERGED_ATOL
        Iterations:   0 - Residuals: 0.00e+00,   NaN 
        --------------- Exiting CG solver ---------------------
      > Iteration   1 - Residuals: 2.58e-09,   7.09e-03 
        --------------- Starting CG solver --------------------
          > Iteration   0 - Residuals: 1.00e+00,   1.00e+00 
          > Iteration   1 - Residuals: 8.18e-01,   8.18e-01 
          > Iteration   2 - Residuals: 7.51e-05,   7.51e-05 
          > Iteration   3 - Residuals: 3.02e-16,   3.02e-16 
        Solver CG finished with reason SOLVER_CONVERGED_RTOL
        Iterations:   3 - Residuals: 3.02e-1

MultiFieldFEFunction():
 num_fields: 2
 num_cells: 3464
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 6857308863574582606

Post-process and tidy up.

In [35]:
mag_integral = sqrt(sum( ∫( uh ⋅ uh )dΩ ))
domain_volume = sum(∫(1)dΩ)
avg_velocity_magnitude = mag_integral / domain_volume

println("Average |u| = ", avg_velocity_magnitude, " m/s. Domain volume = ", domain_volume, " m³.")

writevtk(Ω,"ArcticBoundaryCurrentModel4",cellfields=["uh"=>uh,"ph"=>ph])

Average |u| = 4.9348184245537707e-5 m/s. Domain volume = 0.627342330584225 m³.


(["ArcticBoundaryCurrentModel4.vtu"],)